# PRiSM Track B Closed-Loop (Thin Colab Notebook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/thesis-research-colab/blob/main/PRiSM_trackB_closedloop_simulation_colab.ipynb)

This notebook is intentionally thin. Core logic lives in `src/trackb/*.py`.


## Run Contract

- Pull latest code from GitHub at runtime start.
- Keep this notebook orchestration-only (no large inline modules).
- Persist outputs/checkpoints under Google Drive using `cfg.run_prefix`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Optional for fresh runtimes.
# !pip -q install --upgrade pip
# !pip -q install numpy pandas seaborn scipy tqdm waymo-waymax


In [ ]:
import numpy as np
import pandas as pd

from src.trackb import (
    SearchConfig,
    build_trackb_runner_and_splits,
    configure_persistent_run_prefix,
    export_trackb_artifacts,
    initialize_configs,
    make_waymax_data_iter,
    run_preflight_and_calibration,
    run_surprise_quality_gate,
    run_trackb_closed_loop,
    summarize_method_outputs,
)


In [ ]:
cfg, search_cfg, ckpt_scan_df = initialize_configs()

RUN_TAG = "trackb_closedloop_v1"
PERSIST_ROOT = "/content/drive/MyDrive/prism_trackb_runs"
N_SHARDS = 1
SHARD_ID = 0
RESTORE_FROM_UPLOAD = False

run_prefix = configure_persistent_run_prefix(
    cfg=cfg,
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    shard_id=SHARD_ID,
    n_shards=N_SHARDS,
)
print("run_prefix =", run_prefix)

if len(ckpt_scan_df):
    display(ckpt_scan_df.head(10))


In [ ]:
dataset_config, data_iter = make_waymax_data_iter(cfg)
(
    runner,
    data,
    train_idx,
    test_idx,
    eval_idx_all,
    eval_idx,
    reference_df,
    base_eval_openloop_df,
) = build_trackb_runner_and_splits(
    cfg=cfg,
    data_iter=data_iter,
    dataset_config=dataset_config,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
)

print(f"train={len(train_idx)} test={len(test_idx)} eval_shard={len(eval_idx)}")


In [ ]:
(
    preflight_df,
    closedloop_calib_df,
    trackb_thresholds,
    calib_diag_df,
    calib_quant_df,
) = run_preflight_and_calibration(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
)

print("Calibration thresholds:", trackb_thresholds)
display(preflight_df)
display(calib_diag_df)
display(calib_quant_df)


In [ ]:
gate_summary_df, dist_change_summary_df = run_surprise_quality_gate(
    closedloop_calib_df=closedloop_calib_df,
    surprise_gate_enabled=True,
)
display(gate_summary_df)
display(dist_change_summary_df)


In [ ]:
trackb_results_df, trackb_trace_df = run_trackb_closed_loop(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    thresholds=trackb_thresholds,
    run_prefix=cfg.run_prefix,
    static_frames={
        "base_eval_openloop_df": base_eval_openloop_df,
        "reference_df": reference_df,
        "closedloop_calib_df": closedloop_calib_df,
        "preflight_df": preflight_df,
        "calib_diag_df": calib_diag_df,
        "calib_quant_df": calib_quant_df,
    },
)

print("Result rows:", len(trackb_results_df))
print("Trace rows:", len(trackb_trace_df))


In [ ]:
quick_summary_df, sanity_df, fairness_checks_df, trace_diag_df = summarize_method_outputs(
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
)

artifact_paths = export_trackb_artifacts(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    trackb_results_df=trackb_results_df,
    trackb_trace_df=trackb_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    trackb_thresholds=trackb_thresholds,
    quick_summary_df=quick_summary_df,
    sanity_df=sanity_df,
    fairness_checks_df=fairness_checks_df,
    trace_diag_df=trace_diag_df,
)

display(quick_summary_df)
display(sanity_df)
display(fairness_checks_df)
display(trace_diag_df)

print("Artifacts:")
for key, value in artifact_paths.items():
    print(f" - {key}: {value}")
